# Cursul 3. Surse diverse și preprocesare
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

În C2 am adus datele prin API. Azi le pregătim ca să le putem uni în C4: nume clare pentru coloane și doar rândurile de care avem nevoie.

| Pas | Ce facem | Timp |
|---|---|---|
| 1 | Ce am adus din C2 | 5 min |
| 2 | Numele regiunilor: nomenclatorul NUTS | 10 min |
| 3 | Indicatorii regionali într-un singur tabel | 15 min |
| 4 | SCI pe regiuni: nume clare | 10 min |
| 5 | Migrația: țări și grupuri de țări | 15 min |
| 6 | Echipa: depozitul proiectului | 20 min |

Fiecare pas salvează un fișier în `data/c3`. De la ele pornește C4.

In [ ]:
import requests
import pandas as pd
from pathlib import Path

# Folderul pentru fișierele de azi (dacă există deja, nu se întâmplă nimic)
Path("data/c3").mkdir(exist_ok=True)

## 1. Ce am adus din C2
În `data/c2` avem câte un fișier pe indicator și nivel, salvate cu `cere_eurostat`. **Anticipează:** ce coloane are unul dintre ele?

In [ ]:
densitate = pd.read_csv("data/c2/densitate_nuts3.csv")    # densitatea pe județe, din C2
densitate.head()

Coloanele care ne interesează sunt codul regiunii (`geo`) și `valoare`; celelalte (frecvența, unitatea, anul) sunt la fel pe toate rândurile. Un rând = o regiune. Clujul îl găsim după cod, cu filtrul din C1:

In [ ]:
densitate[densitate["geo"] == "RO113"]      # rândurile în care geo este RO113

**Exercițiu.** Câți locuitori are regiunea Nord-Vest (`RO11`)? Fișierul este `populatie_nuts2.csv`.

In [ ]:
# 1. Citește data/c2/populatie_nuts2.csv în variabila populatie
# 2. Rândul în care geo este "RO11", ca la densitate

Codul `RO11` nu spune singur că e vorba de Nord-Vest. Numele vin din a doua sursă de azi, nomenclatorul NUTS.

## 2. Numele regiunilor: nomenclatorul NUTS
Eurostat publică lista tuturor regiunilor (codul, țara, numele) ca fișier CSV, pe serverul GISCO ([pagina fișierelor](https://gisco-services.ec.europa.eu/distribution/v2/nuts/nuts-2024-files.html)). Îl descărcăm cu `requests`, ca în C2, și îl salvăm în `data/c3`.

In [ ]:
url = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/csv/NUTS_AT_2024.csv"
raspuns = requests.get(url)                                       # cerem fișierul, ca în C2
fisier = Path("data/c3/nuts_gisco.csv")
fisier.write_bytes(raspuns.content)                               # îl salvăm exact cum a venit

nuts = pd.read_csv(fisier)                                        # și îl citim
nuts.head()

Păstrăm trei coloane și le dăm nume în română, aceleași în tot proiectul.

In [ ]:
nuts = nuts[["NUTS_ID", "CNTR_CODE", "NAME_LATN"]]   # codul, țara, numele cu litere latine
nuts.columns = ["regiune", "tara", "denumire"]       # nume noi, în aceeași ordine
nuts.head()

Lungimea codului spune nivelul: `RO` e țara, `RO11` regiunea (NUTS2), `RO113` județul (NUTS3). Nivelul este lungimea codului minus 2.

In [ ]:
nuts["nivel"] = nuts["regiune"].str.len() - 2      # RO -> 0, RO11 -> 2, RO113 -> 3
nuts[nuts["tara"] == "RO"].head(8)

**Exercițiu.** Care sunt regiunile NUTS2 ale României? Două condiții, fiecare între paranteze, legate cu `&`.

In [ ]:
# Rândurile din nuts în care tara este "RO" și nivel este 2

In [ ]:
nuts.to_csv("data/c3/nuts.csv", index=False)

## 3. Indicatorii regionali într-un singur tabel
Avem șase fișiere: trei indicatori, fiecare pe două niveluri. Le punem unul sub altul, într-un singur tabel. Ca să știm de unde vine fiecare rând, adăugăm două coloane: ce măsoară și la ce nivel.

Întâi un singur fișier:

In [ ]:
tabel = pd.read_csv("data/c2/densitate_nuts3.csv")
tabel["indicator"] = "densitate"      # ce măsoară valoarea
tabel["nivel"] = 3                    # 2 = regiuni, 3 = județe
tabel.head()

Aceiași pași pentru fiecare fișier, deci îi punem într-o funcție. Păstrăm doar codul și valoarea și redenumim `geo` în `regiune`, ca în nomenclator.

In [ ]:
def citeste(indicator, nivel):
    """Un fișier din C2, cu două coloane în plus: ce măsoară și la ce nivel."""
    tabel = pd.read_csv(f"data/c2/{indicator}_nuts{nivel}.csv")   # de exemplu data/c2/densitate_nuts3.csv
    tabel = tabel[["geo", "valoare"]]                            # doar codul și valoarea
    tabel["indicator"] = indicator                              # ce măsoară valoarea
    tabel["nivel"] = nivel                                      # 2 = regiuni, 3 = județe
    tabel = tabel.rename(columns={"geo": "regiune"})            # același nume ca în nomenclator
    return tabel

**Exercițiu.** Cu `citeste`, PIB-ul pe locuitor pe județe; apoi rândul Clujului (`RO113`).

In [ ]:
# 1. pib = citeste(...), cu indicatorul "pib_locuitor" și nivelul 3
# 2. rândul în care regiune este "RO113", ca la densitate

Acum toate șase, într-o listă, apoi `pd.concat`, ca la Banca Mondială în C2.

In [ ]:
tabele = [citeste("populatie", 2), citeste("populatie", 3),
          citeste("densitate", 2), citeste("densitate", 3),
          citeste("pib_locuitor", 2), citeste("pib_locuitor", 3)]

regional = pd.concat(tabele)          # cele șase tabele, unul sub altul
regional.shape                        # rânduri, coloane

Clujul apare acum de trei ori, câte un rând pe indicator: tabelul e în format **lung**.

In [ ]:
regional[regional["regiune"] == "RO113"]

**Verificăm:** câte rânduri are fiecare indicator, la fiecare nivel, și câte au valoare? `size` numără rândurile, `count` doar valorile prezente; diferența sunt valorile lipsă.

In [ ]:
regional.groupby(["nivel", "indicator"])["valoare"].agg(["size", "count"])

**Interpretează:** la PIB-ul pe județe lipsesc cele mai multe valori. Ce facem cu ele decidem în C4.

In [ ]:
regional.to_csv("data/c3/regional.csv", index=False)

## 4. SCI pe regiuni: nume clare
SCI există și pe regiuni NUTS2, din aceeași sursă ca `country.csv` din C1 ([HDX](https://data.humdata.org/dataset/social-connectedness-index)). Fișierul stă în depozitul cursului; îl aducem cu `requests`, ca nomenclatorul.

In [ ]:
url = "https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/nuts2_2024.csv"
raspuns = requests.get(url)                       # cerem fișierul, ca nomenclatorul
fisier = Path("data/nuts2_2024.csv")
fisier.write_bytes(raspuns.content)               # îl salvăm exact cum a venit

sci = pd.read_csv(fisier)
sci.head()

Coloanele au nume în engleză. Le dăm nume care spun ce înseamnă în proiect: cele două capete ale perechii.

In [ ]:
nume_noi = {"user_country": "tara_1", "friend_country": "tara_2",
            "user_region": "regiune_1", "friend_region": "regiune_2",
            "scaled_sci": "sci"}
sci = sci.rename(columns=nume_noi)      # se schimbă doar numele din dicționar
sci.head()

**Anticipează:** câte regiuni și câte rânduri?

In [ ]:
print(sci["regiune_1"].nunique(), "regiuni")
print(len(sci), "rânduri")

298 × 298 = 88.804: fiecare regiune cu fiecare, inclusiv cu ea însăși.

**Exercițiu.** Cât este SCI între Nord-Vest (`RO11`) și Lombardia (`ITC4`)? Apoi schimbă ordinea celor două coduri.

In [ ]:
# Rândul din sci în care regiune_1 este "RO11" și regiune_2 este "ITC4"

Aceeași valoare, în ambele ordini: fiecare pereche apare de două ori. În C4 păstrăm o singură apariție.

In [ ]:
sci.to_csv("data/c3/sci_nuts2.csv", index=False)

SCI pe județe (NUTS3) are 1,8 milioane de rânduri, peste 40 MB. Nu îl folosim la curs; echipele care au nevoie de el îl descarcă de pe [HDX](https://data.humdata.org/dataset/social-connectedness-index) și fac aceiași pași.

## 5. Migrația: țări și grupuri de țări
Deschidem tabelul din C2. O capcană: codul Namibiei este `NA`, iar pandas citește `NA` ca valoare lipsă. Îi spunem să considere lipsă doar celulele goale.

In [ ]:
migratie = pd.read_csv("data/c2/migratie.csv", keep_default_na=False, na_values=[""])
migratie["citizen"].unique()

**Anticipează:** care coduri nu sunt țări? Eurostat pune în aceeași coloană și grupuri: `EU27_2020_FOR` (străinii din UE), `FOR` (toți străinii), `NAT` (cetățenii proprii), `TOTAL`. Dacă le păstrăm, aceiași oameni sunt numărați de mai multe ori.

Codurile de țară au exact două litere. Păstrăm rândurile în care **ambele** coloane sunt țări.

In [ ]:
cetatenie_tara = migratie["citizen"].str.len() == 2     # True unde cetățenia e o țară
rezidenta_tara = migratie["geo"].str.len() == 2         # True unde locul e o țară
migratie = migratie[cetatenie_tara & rezidenta_tara]    # amândouă

Scoatem și cetățenii propriei țări (românii din România): nu sunt migranți. Apoi păstrăm trei coloane, cu nume clare.

In [ ]:
migratie = migratie[migratie["citizen"] != migratie["geo"]]      # cetățenia diferă de țară
migratie = migratie[["geo", "citizen", "valoare"]]               # coloanele, în ordinea dorită
migratie.columns = ["tara_rezidenta", "cetatenie", "persoane"]   # nume clare
migratie.head()

Un rând = câte persoane cu o cetățenie trăiesc într-o țară. Exemplul nostru: românii, după țara în care trăiesc.

In [ ]:
romani = migratie[migratie["cetatenie"] == "RO"]
romani.sort_values("persoane", ascending=False).head(10)

In [ ]:
migratie.to_csv("data/c3/migratie.csv", index=False)

**Exercițiul echipei.** Alegeți o cetățenie din coridorul vostru și afișați primele 10 țări de rezidență. Lipsește o țară pe care v-ați fi așteptat să o vedeți? (Nu toate țările raportează; revenim în C4.)

In [ ]:
# Ca la români: rândurile cu cetățenia aleasă, ordonate după persoane, primele 10

## 6. Echipa: depozitul proiectului
De azi lucrați și în echipă. Fiecare echipă are pe GitHub depozitul ei, `dsw-ubb-2026/echipa-N`, cu aceeași structură pentru toate echipele. Invitația a venit pe emailul contului GitHub.

### 6.1. Depozitul pe calculator (fiecare membru)
1. Acceptați invitația: `https://github.com/dsw-ubb-2026/echipa-N/invitations` (N = numărul echipei).
2. În terminalul din VS Code, în folderul `dsw/` (nu în `curs/`):
```bash
git clone https://github.com/dsw-ubb-2026/echipa-N.git proiect
```
Prima dată pe un calculator, Git vă cere numele și emailul:
```bash
git config --global user.name "Prenume Nume"
git config --global user.email "emailul-de-pe-github"
```
Rezultatul:
```text
dsw/
├── .venv/
├── curs/                 caietele de curs, fără Git
└── proiect/              depozitul echipei, cu Git
    ├── data/raw/         datele originale, neschimbate
    ├── data/processed/   tabelele produse de caiete
    ├── notebooks/        câte un caiet pe pas
    ├── src/              cod comun, în fișiere .py
    ├── figures/          figurile pentru raport
    ├── docs/             raportul
    └── README.md         descrierea proiectului
```

### 6.2. Datele proiectului
Datele de la curs rămân în `curs/data`. Datele proiectului stau în `proiect/data`: în `raw/` ce aduceți de la sursă, exact cum a venit; în `processed/` ce produc caietele voastre. Fișierele de peste 50 MB (de exemplu SCI pe județe) nu se urcă pe GitHub: caietul care le folosește spune de unde se descarcă.

### 6.3. Primul commit al echipei
Un singur membru completează în `README.md`, provizoriu: titlul, întrebarea de cercetare, unitatea de analiză și tabelul surselor de date. Apoi:
```bash
cd proiect
git add README.md
git commit -m "Tema si intrebarea provizorie"
git push
```
Ceilalți aduc schimbarea:
```bash
cd proiect
git pull
```
`commit` salvează pasul pe calculatorul vostru, `push` îl trimite pe GitHub, `pull` îl aduce de la colegi. Regula care evită conflictele: fiecare lucrează în caietul lui și dă `git pull` înainte să înceapă.

### 6.4. Ce stabiliți azi în echipă
- tema proiectului și o întrebare provizorie: coridorul sau țara pe care o studiați;
- un rol provizoriu pentru fiecare: colectarea, preprocesarea, curățarea și integrarea, explorarea, modelul. Rolurile se pot schimba, iar toți pot colabora la orice pas;
- ce reprezintă un rând în tabelul vostru final;
- o problemă de date de rezolvat în C4.

Tema 1, pe Teams, cere fiecăruia să descrie cu propriile cuvinte tema proiectului, datele și rolul ales.

## Ce ducem mai departe
În `data/c3` avem `nuts.csv`, `regional.csv`, `sci_nuts2.csv` și `migratie.csv`: aceleași nume de coloane, gata de unit în C4. Echipa are depozitul pe calculator și primul commit pe GitHub.

**Poți explica fără cod?** Ce reprezintă un rând în fiecare tabel? De ce adăugăm coloanele `indicator` și `nivel`? De ce `NA` nu e întotdeauna o valoare lipsă? De ce scoatem grupurile de țări? Ce diferență este între `commit`, `push` și `pull`?

### Surse
- [GISCO: fișierele NUTS 2024](https://gisco-services.ec.europa.eu/distribution/v2/nuts/nuts-2024-files.html)
- [Social Connectedness Index, pe HDX](https://data.humdata.org/dataset/social-connectedness-index)
- [Eurostat: populația după cetățenie, `migr_pop1ctz`](https://ec.europa.eu/eurostat/databrowser/view/migr_pop1ctz/default/table)
- [Git: înregistrarea schimbărilor](https://git-scm.com/book/en/v2/Git-Basics-Recording-Changes-to-the-Repository)